QUESTION 10


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))

img = Image.open(filename).convert("RGB")
arr = np.array(img)
h, w = arr.shape[:2]

# Transformation matrices
A1 = np.array([[2.0, 0.0],
               [0.0, 0.5]])

A2 = np.array([[0.0, -1.0],
               [1.0,  0.0]])

A3 = np.array([[1.0, 1.0],
               [0.0, 1.0]])

A4 = np.array([[-1.0, 0.0],
               [0.0, 1.0]])

A5 = np.array([[1.0, 0.0],
               [0.0, 0.0]])

matrices = {
    "A1 Scaling": A1,
    "A2 90-degree Rotation": A2,
    "A3 Horizontal Shear": A3,
    "A4 Reflection in y-axis": A4,
    "A5 Projection onto x-axis": A5
}

def transform_image(image_array, A):
    """Apply a 2-D linear transformation about the image centre."""
    h, w = image_array.shape[:2]

    # Coordinates are [x, y]
    yy, xx = np.indices((h, w))
    x = xx - (w - 1) / 2
    y = (h - 1) / 2 - yy

    # mapping: [x_new, y_new]^T = A [x, y]^T
    pts = np.vstack((x.ravel(), y.ravel()))
    new_pts = A @ pts

    x_new = new_pts[0]
    y_new = new_pts[1]

    # Keep output canvas the same size and round to nearest pixel.
    col = np.rint(x_new + (w - 1) / 2).astype(int)
    row = np.rint((h - 1) / 2 - y_new).astype(int)

    valid = (col >= 0) & (col < w) & (row >= 0) & (row < h)


    yy_out, xx_out = np.indices((h, w))
    x_out = xx_out - (w - 1) / 2
    y_out = (h - 1) / 2 - yy_out
    out_pts = np.vstack((x_out.ravel(), y_out.ravel()))

    det = np.linalg.det(A)
    if abs(det) > 1e-10:
        invA = np.linalg.inv(A)
        src_pts = invA @ out_pts
        sx = np.rint(src_pts[0] + (w - 1) / 2).astype(int)
        sy = np.rint((h - 1) / 2 - src_pts[1]).astype(int)
        valid2 = (sx >= 0) & (sx < w) & (sy >= 0) & (sy < h)

        output = np.zeros_like(image_array)
        output_flat = output.reshape(-1, 3)
        source_flat = image_array[sy[valid2], sx[valid2]]
        output_flat[valid2] = source_flat
        return output

    # A5 is singular.
    output = np.zeros_like(image_array)
    out_flat = output.reshape(-1, 3)
    out_flat[col[valid] + row[valid] * w] = image_array.reshape(-1, 3)[valid]

    return output

def rank(A):
    return np.linalg.matrix_rank(A)

print("Original image size:", (w, h))

for name, A in matrices.items():
    print("\n" + "="*55)
    print(name)
    print("Matrix:\n", A)
    print("T(e1) =", A @ np.array([1, 0]))
    print("T(e2) =", A @ np.array([0, 1]))
    print("rank(A) =", rank(A))

    if rank(A) < 2:
        print("Information/dimension lost: YES")
    else:
        print("Information/dimension lost: NO")

    transformed = transform_image(arr, A)

    plt.figure(figsize=(7, 5))
    plt.imshow(transformed)
    plt.axis("off")
    plt.title(name)
    plt.show()

# For A5 the true matrix is [[1,0],[0,0]], so all points are projected
print("\nA5 true matrix =")
print(A5)
print("A5 is rank 1, so one coordinate/dimension is lost.")




# ques

QUESTION 11


In [ ]:
# Q11 — Interactive Image Transformation Toolbox
# SMAS-1 Assignment 3 — Linear Transformations
# Google Colab version

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files
from IPython.display import display

print("Upload a JPG/PNG image.")
uploaded = files.upload()
filename = next(iter(uploaded))
original = Image.open(filename).convert("RGB")
img = original.copy()

def show_image(im, title="Image"):
    plt.figure(figsize=(8, 5))
    plt.imshow(im)
    plt.axis("off")
    plt.title(title)
    plt.show()

def rotate_image(im, angle):
    # PIL uses degrees counterclockwise.
    return im.rotate(angle, expand=True)

def resize_image(im, factor):
    if factor <= 0:
        raise ValueError("Resize factor must be greater than 0.")
    new_size = (max(1, int(im.width * factor)),
                max(1, int(im.height * factor)))
    return im.resize(new_size)

def flip_image(im, direction):
    if direction == "horizontal":
        return im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    elif direction == "vertical":
        return im.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
    else:
        raise ValueError("Direction must be horizontal or vertical.")

def shear_image(im, shear):
    # Horizontal shear: x' = x + k*y
    w, h = im.size
    k = float(shear)
    shift = abs(k) * h
    new_w = int(np.ceil(w + shift))

    # PIL affine transform expects source coordinates:
    # x = x' - k*y'
    if k >= 0:
        a = 1
        b = -k
        c = 0
    else:
        a = 1
        b = -k
        c = -k * h

    return im.transform(
        (new_w, h),
        Image.Transform.AFFINE,
        (a, b, c, 0, 1, 0),
        resample=Image.Resampling.BICUBIC
    )

def custom_matrix(im, A):
    """Apply a 2x2 matrix around the image centre using inverse mapping."""
    arr = np.array(im)
    h, w = arr.shape[:2]

    yy, xx = np.indices((h, w))
    x = xx - (w - 1) / 2
    y = (h - 1) / 2 - yy
    target = np.vstack((x.ravel(), y.ravel()))

    det = np.linalg.det(A)
    if abs(det) < 1e-10:
        raise ValueError("Custom matrix must be invertible for this toolbox.")

    source = np.linalg.inv(A) @ target
    sx = np.rint(source[0] + (w - 1) / 2).astype(int)
    sy = np.rint((h - 1) / 2 - source[1]).astype(int)

    valid = (sx >= 0) & (sx < w) & (sy >= 0) & (sy < h)

    out = np.zeros_like(arr).reshape(-1, 3)
    out[valid] = arr[sy[valid], sx[valid]]
    return Image.fromarray(out.reshape(h, w, 3))

while True:
    print("\n========== IMAGE TRANSFORMATION TOOLBOX ==========")
    print("1. Rotate")
    print("2. Resize")
    print("3. Flip")
    print("4. Shear")
    print("5. Custom Matrix")
    print("6. Reset")
    print("7. Exit")

    choice = input("""Enter your choice (1-7):
1:rotate_image
2:resize
3:flip
4:horizontal_shear
5:custom_matrix
6:reset
7:exit""").strip()

    try:
        if choice == "1":
            angle = float(input("Enter rotation angle in degrees: "))
            img = rotate_image(img, angle)
            show_image(img, f"Rotated by {angle}°")

        elif choice == "2":
            factor = float(input("Enter resize factor (e.g. 2 or 0.5): "))
            img = resize_image(img, factor)
            show_image(img, f"Resized by factor {factor}")

        elif choice == "3":
            direction = input("Enter flip direction (horizontal/vertical): ").strip().lower()
            img = flip_image(img, direction)
            show_image(img, f"Flipped {direction}")

        elif choice == "4":
            shear = float(input("Enter horizontal shear factor k: "))
            img = shear_image(img, shear)
            show_image(img, f"Horizontal shear, k={shear}")

        elif choice == "5":
            print("Enter the four entries of a 2x2 matrix.")
            a = float(input("a11 = "))
            b = float(input("a12 = "))
            c = float(input("a21 = "))
            d = float(input("a22 = "))
            A = np.array([[a, b], [c, d]], dtype=float)

            print("\nCustom matrix:")
            print(A)
            print("T(e1) =", A @ np.array([1, 0]))
            print("T(e2) =", A @ np.array([0, 1]))
            print("rank(A) =", np.linalg.matrix_rank(A))

            img = custom_matrix(img, A)
            show_image(img, "Custom Matrix Transformation")

        elif choice == "6":
            img = original.copy()
            show_image(img, "Reset to Original")

        elif choice == "7":
            print("Toolbox closed.")
            break

        else:
            print("Please enter a number from 1 to 7.")

    except Exception as e:
        print("Error:", e)